In [1]:
import sys
import warnings 
from pathlib import Path
from collections import defaultdict
sys.path.append(str(Path.cwd().parents[1]))

import pandas as pd

from source.exp_functions import CVTracker
from training import FMAP, DATASETS
from configs.uci import CV_NAME, RES_NAME, PRED_NAME

warnings.filterwarnings("ignore")

def get_res_df(model: str, hess: str, data_list: list[str]):
    dd = defaultdict(list)
    for data in data_list:
        dd['data'].append(data if data != "wine_red" else "red wine")
        res_dir = Path(f'./artifacts/training_artifacts/{model}/{data}/{hess}/ll/{FMAP}/') 
        try:
            tracker = CVTracker(res_dir, RES_NAME, CV_NAME, PRED_NAME)
            res_df, _, _ = tracker.load()
            res_str = f"{res_df['nll_test'].mean():.2f}±{res_df['nll_test'].std():.2f}"
            dd[model.upper()].append(res_str)
        except:
            dd[model.upper()].append('N/A±0.00')
    return pd.DataFrame(dd).rename({'LA_BTN': 'LA-TNKM'}, axis=1)

In [2]:
uci_gwi = pd.read_csv('./extra_data/uci_gwi_paper_table.csv', index_col=0)
for model, hess in zip(['mf_btn', 'sp_btn', 'la_btn'], ['mf', 'mf', 'last']):
    res_df = get_res_df(model, hess, DATASETS)
    uci_gwi = pd.merge(uci_gwi, res_df, left_on='Dataset', right_on='data')
    uci_gwi = uci_gwi.drop('data', axis=1)
uci_gwi['Dataset'] = uci_gwi['Dataset'].apply(lambda x: x.upper())
uci_gwi = uci_gwi.drop(['FBNN'], axis=1).rename({'GWI DNN-SVGP': 'GWI-DNN'}, axis=1)
display(uci_gwi)

,Dataset,N,D,GWI-DNN,FVI,VIP-BNN,BBB,α=0.5,EXACT GP,MF_BTN,SP_BTN,LA-TNKM
0,BOSTON,506,13,2.27±0.06,2.33±0.04,2.45±0.04,2.76±0.04,2.45±0.02,2.46±0.04,1.45±0.11,1.19±0.07,0.95±0.16
1,CONCRETE,1030,8,2.64±0.06,2.88±0.06,3.02±0.02,3.28±0.01,3.06±0.03,3.05±0.02,1.41±0.07,0.74±0.06,0.82±0.09
2,ENERGY,768,8,0.91±0.12,0.58±0.05,0.56±0.04,2.17±0.02,0.95±0.09,0.54±0.02,1.41±0.04,0.62±0.07,-1.40±0.04
3,KIN8NM,8192,8,-1.2±0.03,-1.15±0.01,-1.12±0.01,-0.81±0.01,-0.92±0.02,N/A±0.00,1.42±0.01,0.89±0.02,0.48±0.02
4,NAVAL,11934,16,-6.76±0.1,-7.21±0.06,-5.62±0.04,-2.80±0.00,-2.97±0.14,N/A±0.00,1.42±0.01,1.37±0.02,-1.16±0.64
5,POWER,9568,4,2.74±0.02,2.69±0.00,2.92±0.00,2.83±0.01,2.81±0.00,N/A±0.00,1.42±0.02,0.57±0.01,-0.01±0.03
6,PROTEIN,45730,9,2.87±0.0,2.85±0.00,2.87±0.00,3.00±0.00,2.90±0.00,N/A±0.00,N/A±0.00,N/A±0.00,1.06±0.01
7,RED WINE,1588,11,0.76±0.08,0.97±0.06,0.97±0.02,1.01±0.02,1.01±0.02,0.26±0.03,1.35±0.06,1.26±0.05,1.24±0.06
8,YACHT,308,6,0.29±0.1,0.59±0.11,-0.02±0.07,1.11±0.04,0.79±0.11,0.10±0.05,1.38±0.10,0.58±0.34,-0.52±0.17


In [3]:
caption = (
    r"The average test NLL on several UCI regression datasets. "
    + r"We train on random 90\% of the data and predict on 10\%. "
    + r"This is repeated 10 times and we report mean and standard deviation. "
    + r"$N$ is the sample size; $D$ is the data dimensionality. "
    + r"Among the evaluated methods, LA-TNKM (Last) ranks highest on five of the nine datasets and performs comparably on the rest."
)
print(
    (
        uci_gwi
        .to_latex(
            float_format="%.2f", 
            column_format='||l|c|c||c|c|c|c|c|c|c|c||c||',
            caption=caption,
            label="table:uci-comparison",
            multirow=False,
            index=False,
        ).replace('_', '-')
    )
);

\begin{table}
\caption{The average test NLL on several UCI regression datasets. We train on random 90\% of the data and predict on 10\%. This is repeated 10 times and we report mean and standard deviation. $N$ is the sample size; $D$ is the data dimensionality. Among the evaluated methods, LA-TNKM (Last) ranks highest on five of the nine datasets and performs comparably on the rest.}
\label{table:uci-comparison}
\begin{tabular}{||l|c|c||c|c|c|c|c|c|c|c||c||}
\toprule
Dataset & N & D & GWI-DNN & FVI & VIP-BNN & BBB & α=0.5 & EXACT GP & MF-BTN & SP-BTN & LA-TNKM \\
\midrule
BOSTON & 506 & 13 & 2.27±0.06 & 2.33±0.04 & 2.45±0.04 & 2.76±0.04 & 2.45±0.02 & 2.46±0.04 & 1.45±0.11 & 1.19±0.07 & 0.95±0.16 \\
CONCRETE & 1030 & 8 & 2.64±0.06 & 2.88±0.06 & 3.02±0.02 & 3.28±0.01 & 3.06±0.03 & 3.05±0.02 & 1.41±0.07 & 0.74±0.06 & 0.82±0.09 \\
ENERGY & 768 & 8 & 0.91±0.12 & 0.58±0.05 & 0.56±0.04 & 2.17±0.02 & 0.95±0.09 & 0.54±0.02 & 1.41±0.04 & 0.62±0.07 & -1.40±0.04 \\
KIN8NM & 8192 & 8 & -1.2±0.03 & 